# Training a "Widish" ResNet on Tiny ImageNet

## What This Notebook Does

This notebook trains an image classification model to recognize **200 different categories** of objects in small (64x64 pixel) images. We use a dataset called **Tiny ImageNet**, which is a smaller version of the famous ImageNet dataset.

The model we build is called a **"Widish" ResNet** - it's a neural network that is moderately wide (has many channels/filters) rather than extremely deep (many layers). This is a balance between the "standard" and "wide" approaches.

## What You'll Learn

1. **How to load and prepare image data** for training
2. **How to build custom PyTorch datasets** from scratch
3. **Data augmentation techniques** that help prevent overfitting
4. **ResNet architecture** - the building blocks of modern image classifiers
5. **Pre-activation ResNets** - an improved version of ResNets
6. **Training with modern techniques** like OneCycleLR and mixed precision

## Prerequisites

- Basic Python (variables, functions, classes, loops)
- Understanding that neural networks take inputs and produce outputs
- No deep learning experience required - we'll explain everything!

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library.

**GPU note:** Training cells need a GPU. **Runtime &rarr; GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
!pip install -q fastcore fastai diffusers datasets torcheval accelerate wandb
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))
try:
    import miniai
    print(f'miniai loaded from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`24_imgnet_tiny-widish_explained.ipynb`), unchanged.*

---

---
## Section 1: Setting Up the Environment

Before we can train our model, we need to import the necessary libraries and configure our environment. Think of this as gathering all our tools before starting a project.

### 1.1 Selecting the GPU

Modern deep learning requires a lot of computation. GPUs (Graphics Processing Units) can do these calculations much faster than CPUs. If you have multiple GPUs, you can choose which one to use.

In [ ]:
# Import the 'os' module which lets us interact with the operating system
import os

# Set an environment variable to tell PyTorch which GPU to use
# '2' means "use the third GPU" (counting starts from 0)
# If you only have one GPU, change this to '0'
# IMPORTANT: This line MUST come BEFORE importing torch!
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

**What is an environment variable?**

Environment variables are like global settings that programs can read. When we set `CUDA_VISIBLE_DEVICES`, we're telling PyTorch "only see GPU number 2, pretend the others don't exist."

**Why do we need to set this before importing torch?**

When PyTorch loads (via `import torch`), it immediately scans for available GPUs. If we set the environment variable after importing, PyTorch has already made its decision about which GPUs to use.

### 1.2 Importing Libraries

Now we import all the tools we'll need. Don't worry about understanding every import right now - we'll explain each one as we use it.

In [ ]:
# ============================================================
# STANDARD PYTHON LIBRARIES
# ============================================================

# shutil: For file operations like copying and extracting archives
import shutil

# timm: "PyTorch Image Models" - a library of pre-built image models
import timm

# os: Operating system utilities (file paths, environment variables)
import os

# torch: PyTorch - the deep learning framework we're using
import torch

# random: For generating random numbers
import random

# datasets: HuggingFace datasets library (we may use this for other datasets)
import datasets

# math: Mathematical functions like sqrt, sin, cos, etc.
import math

# warnings: To control warning messages
import warnings

# ============================================================
# DATA MANIPULATION AND VISUALIZATION
# ============================================================

# fastcore: A library of utilities that makes Python coding easier
# We import it "as fc" so we can write fc.something instead of fastcore.all.something
import fastcore.all as fc

# numpy: The fundamental library for numerical computing in Python
# We import it "as np" (a common convention)
import numpy as np

# matplotlib: The main plotting library for Python
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================================================
# DEEP LEARNING SPECIFIC
# ============================================================

# k_diffusion: A library for diffusion models (we use some utilities from it)
import k_diffusion as K

# torchvision.transforms: Image transformation functions (resize, crop, flip, etc.)
import torchvision.transforms as T

# torchvision.transforms.functional: Same transforms but as functions (not classes)
import torchvision.transforms.functional as TF

# torch.nn.functional: Neural network functions (activation functions, loss functions)
import torch.nn.functional as F

# ============================================================
# SPECIFIC IMPORTS (from within libraries)
# ============================================================

# DataLoader: Loads data in batches for training
# default_collate: Combines individual samples into a batch
from torch.utils.data import DataLoader, default_collate

# Path: A convenient way to work with file paths
from pathlib import Path

# init: Weight initialization functions
from torch.nn import init

# L: A list class with extra features from fastcore
from fastcore.foundation import L

# nn: Neural network building blocks (layers, modules)
# tensor: Creates a PyTorch tensor (like a numpy array but can run on GPU)
from torch import nn, tensor

# itemgetter: Efficiently gets items from collections
from operator import itemgetter

# MulticlassAccuracy: Calculates accuracy for classification tasks
from torcheval.metrics import MulticlassAccuracy

# partial: Creates a new function with some arguments pre-filled
from functools import partial

# lr_scheduler: Learning rate scheduling (changes learning rate during training)
from torch.optim import lr_scheduler

# optim: Optimizers (algorithms that update model weights)
from torch import optim

# read_image: Fast image loading function
# ImageReadMode: Specifies how to read the image (RGB, grayscale, etc.)
from torchvision.io import read_image, ImageReadMode

# glob: Finds files matching a pattern (like *.jpg)
from glob import glob

# ============================================================
# MINIAI MODULES (Custom framework for this course)
# ============================================================

# These are custom modules built in earlier notebooks of this course
from miniai.datasets import *    # Dataset utilities
from miniai.conv import *        # Convolution helpers
from miniai.learner import *     # Training loop
from miniai.activations import * # Activation functions
from miniai.init import *        # Weight initialization
from miniai.sgd import *         # Optimizers
from miniai.resnet import *      # ResNet building blocks
from miniai.augment import *     # Data augmentation
from miniai.accel import *       # Acceleration (mixed precision)
from miniai.training import *    # Training utilities

**Understanding `import X as Y`:**

When we write `import numpy as np`, we're saying:
- "Import the numpy library"
- "But let me refer to it as `np` instead of `numpy`"

This is just for convenience - `np.array([1,2,3])` is shorter than `numpy.array([1,2,3])`.

**Understanding `from X import Y`:**

When we write `from torch import nn, tensor`, we're saying:
- "From the torch library, import just the `nn` module and `tensor` function"
- Now we can use `nn.Linear(...)` instead of `torch.nn.Linear(...)`

**Understanding `from X import *`:**

The asterisk (`*`) means "import everything". So `from miniai.datasets import *` imports all functions and classes from that module. This is convenient but can make it unclear where a function came from.

In [ ]:
# Import progress bar for showing training progress
from fastprogress import progress_bar

### 1.3 Configuring Display and Reproducibility

In [ ]:
# Configure how PyTorch prints tensors (numbers)
# precision=5: Show 5 decimal places
# linewidth=140: Allow up to 140 characters per line before wrapping
# sci_mode=False: Don't use scientific notation (1e-5), show full numbers
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)

# Set a random seed for PyTorch
# This makes random operations reproducible (you get the same "random" numbers each time)
torch.manual_seed(1)

# Set the resolution for matplotlib figures (70 dots per inch)
mpl.rcParams['figure.dpi'] = 70

# Set random seeds for ALL random number generators (numpy, python, torch)
# This ensures complete reproducibility
set_seed(42)

# Limit the number of CPU workers for data loading
# Too many workers can cause memory issues or slow things down
if fc.defaults.cpus > 8: 
    fc.defaults.cpus = 8

**What is reproducibility and why does it matter?**

Deep learning involves many random operations:
- Random weight initialization
- Random shuffling of training data
- Random data augmentation

By setting a "seed" (a starting point for the random number generator), we ensure that:
1. We get the same results each time we run the code
2. Others can reproduce our experiments exactly
3. We can fairly compare different models

Think of it like this: if you and a friend both start with seed 42, you'll both generate the same sequence of "random" numbers.

---
## Section 2: Loading the Tiny ImageNet Dataset

Now we'll download and prepare our training data. **Tiny ImageNet** is a dataset of 64x64 pixel images organized into 200 categories (like "goldfish", "school bus", "banana", etc.).

### 2.1 Understanding the Dataset

Before we load the data, let's understand what we're working with:

| Property | Value |
|----------|-------|
| Image size | 64 x 64 pixels |
| Number of classes | 200 |
| Training images | 100,000 (500 per class) |
| Validation images | 10,000 (50 per class) |
| File format | JPEG |
| Total download size | ~400 MB |

In [ ]:
# ============================================================
# STEP 1: Create a folder to store our data
# ============================================================

# Path('data') creates a Path object pointing to a folder called 'data'
# Path objects are easier to work with than plain strings for file operations
path_data = Path('data')

# Create the folder if it doesn't exist
# exist_ok=True means "don't raise an error if it already exists"
path_data.mkdir(exist_ok=True)

# The full path to Tiny ImageNet (inside the data folder)
# The '/' operator with Path objects joins paths (like os.path.join)
path = path_data / 'tiny-imagenet-200'

# ============================================================
# STEP 2: Download and extract the dataset (if not already done)
# ============================================================

# URL where the dataset is hosted (Stanford's CS231n course website)
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'

# Only download if we haven't already
if not path.exists():
    # Download the zip file to our data folder
    # fc.urlsave downloads a file from a URL and saves it
    path_zip = fc.urlsave(url, path_data)
    
    # Extract the zip file
    # shutil.unpack_archive extracts any archive format (zip, tar, etc.)
    shutil.unpack_archive('data/tiny-imagenet-200.zip', 'data')

# ============================================================
# STEP 3: Set our batch size
# ============================================================

# Batch size: how many images to process at once
# Larger batches = faster training but use more GPU memory
# 512 works well for 64x64 images on a modern GPU (8GB+ VRAM)
bs = 512

**What is a batch size?**

Instead of showing the neural network one image at a time, we show it many images at once (a "batch"). This is more efficient because:

1. GPUs are designed to process many things in parallel
2. It provides more stable gradient estimates

Think of it like grading papers: it's more efficient to grade all of Chapter 1 for everyone, then all of Chapter 2, rather than grading one student's entire test before moving to the next.

**Trade-offs:**
- Larger batch = faster per epoch, but may need lower learning rate
- Smaller batch = slower per epoch, but can use higher learning rate
- Batch size is limited by GPU memory

### 2.2 Dataset Folder Structure

Understanding how the data is organized helps us write code to load it:

```
tiny-imagenet-200/
|
+-- train/                          <- Training images
|   +-- n01443537/                  <- Class folder (WordNet ID)
|   |   +-- images/                 <- Actual images
|   |   |   +-- n01443537_0.JPEG
|   |   |   +-- n01443537_1.JPEG
|   |   |   +-- ... (500 images)
|   |   +-- n01443537_boxes.txt     <- Bounding box info (we don't use this)
|   +-- n01629819/                  <- Another class
|   +-- ... (200 class folders)
|
+-- val/                            <- Validation images
|   +-- images/                     <- ALL validation images in ONE folder
|   |   +-- val_0.JPEG
|   |   +-- val_1.JPEG
|   |   +-- ... (10,000 images)
|   +-- val_annotations.txt         <- File mapping image names to classes
|
+-- wnids.txt                       <- List of the 200 class IDs
+-- words.txt                       <- Maps class IDs to English names
```

**Important observations:**
1. Training images: The class is encoded in the folder name (parent.parent of the image)
2. Validation images: All in one folder, class info in a separate text file

### 2.3 Creating a Custom Dataset for Training Data

PyTorch needs data in a specific format. We create a "Dataset" class that knows how to:
1. Find all the images
2. Return any image by its index number
3. Report how many images there are

In [ ]:
class TinyDS:
    """
    A PyTorch-compatible Dataset for Tiny ImageNet training data.
    
    A Dataset must have:
    - __len__: Returns the total number of samples
    - __getitem__: Returns a single sample by index
    
    For training data, the class label is encoded in the folder structure:
    train/n01443537/images/n01443537_0.JPEG
          ^^^^^^^^^^ This is the class ID
    """
    
    def __init__(self, path):
        """
        Initialize the dataset.
        
        Args:
            path: Path to the training folder (e.g., 'data/tiny-imagenet-200/train')
        """
        # Convert to Path object for easier path manipulation
        self.path = Path(path)
        
        # Find ALL JPEG files in all subdirectories
        # glob pattern explanation:
        #   str(path/'**/*.JPEG') creates a string like 'data/.../train/**/*.JPEG'
        #   **  = match any number of subdirectories (including zero)
        #   *   = match any filename
        #   .JPEG = files must end with .JPEG
        #   recursive=True = allow ** to match multiple directory levels
        self.files = glob(str(path / '**/*.JPEG'), recursive=True)
    
    def __len__(self): 
        """
        Return the total number of images in the dataset.
        
        This is called when you do len(dataset).
        """
        return len(self.files)
    
    def __getitem__(self, i): 
        """
        Return a single sample (image path and class ID) by index.
        
        This is called when you do dataset[i].
        
        Args:
            i: The index of the sample to retrieve (0 to len-1)
        
        Returns:
            A tuple of (file_path, class_id)
            - file_path: String path to the image file
            - class_id: WordNet ID like 'n01443537'
        """
        # Get the file path at index i
        file_path = self.files[i]
        
        # Extract the class ID from the path
        # Example path: .../train/n01443537/images/n01443537_0.JPEG
        #                        ^^^^^^^^^^  <-- this is what we want
        # 
        # Path(file_path) converts string to Path object
        # .parent = the containing folder ("images")
        # .parent.parent = the folder above that ("n01443537")
        # .name = just the folder name, not the full path
        class_id = Path(file_path).parent.parent.name
        
        return file_path, class_id

# Create the training dataset
# We pass the path to the 'train' folder
tds = TinyDS(path / 'train')

**Let's understand this step by step with an example:**

```python
# When we create the dataset:
tds = TinyDS('data/tiny-imagenet-200/train')

# Inside __init__, glob finds all JPEG files:
self.files = [
    'data/tiny-imagenet-200/train/n01443537/images/n01443537_0.JPEG',
    'data/tiny-imagenet-200/train/n01443537/images/n01443537_1.JPEG',
    'data/tiny-imagenet-200/train/n01629819/images/n01629819_0.JPEG',
    ... # 100,000 files total
]

# When we access tds[0]:
file_path = 'data/tiny-imagenet-200/train/n01443537/images/n01443537_0.JPEG'
class_id = 'n01443537'  # Extracted from the folder structure
# Returns: ('data/.../n01443537_0.JPEG', 'n01443537')
```

### 2.4 Creating a Dataset for Validation Data

Validation images are organized differently - all in one folder with a text file containing the labels. We need to read that text file first.

In [ ]:
# Path to the annotations file
path_anno = path / 'val' / 'val_annotations.txt'

# Read and parse the annotations file
# The file format is: filename<TAB>class_id<TAB>x<TAB>y<TAB>w<TAB>h
# We only need the first two columns (filename and class_id)
#
# Let's break this down:
#   path_anno.read_text()           -> Reads the entire file as a string
#   .splitlines()                   -> Splits into a list of lines
#   for o in ...                    -> Loop through each line
#   o.split('\t')                   -> Split each line by tab character
#   [:2]                            -> Take only first 2 items (filename, class)
#   dict(...)                       -> Convert list of pairs to dictionary

anno = dict(o.split('\t')[:2] for o in path_anno.read_text().splitlines())

**Example of what the annotations file looks like:**

```
val_0.JPEG    n03444034    0    32    64    64
val_1.JPEG    n04067472    20   14    44    50
val_2.JPEG    n04070727    0    0     64    64
```

Each line has:
- Filename (val_0.JPEG)
- Class ID (n03444034)
- Bounding box coordinates (we ignore these)

**After parsing, our `anno` dictionary looks like:**

```python
{
    'val_0.JPEG': 'n03444034',
    'val_1.JPEG': 'n04067472',
    'val_2.JPEG': 'n04070727',
    # ... 10,000 entries
}
```

In [ ]:
class TinyValDS(TinyDS):
    """
    Dataset for Tiny ImageNet validation data.
    
    Inherits from TinyDS (gets the same __init__ and __len__).
    Only overrides __getitem__ because we get the class differently.
    
    Inheritance means:
    - TinyValDS automatically has everything TinyDS has
    - We only need to write the parts that are different
    """
    
    def __getitem__(self, i): 
        """
        Return a validation sample by index.
        
        Unlike training data, validation images are all in one folder.
        We look up the class in our annotations dictionary.
        """
        # Get the file path
        file_path = self.files[i]
        
        # Get just the filename (e.g., 'val_0.JPEG')
        # os.path.basename extracts the filename from a full path
        filename = os.path.basename(file_path)
        
        # Look up the class ID in our annotations dictionary
        class_id = anno[filename]
        
        return file_path, class_id

In [ ]:
# Create the validation dataset
vds = TinyValDS(path / 'val')

---
## Section 3: Transforming the Data

Our datasets currently return:
- x: A file path (string)
- y: A class ID (string like 'n01443537')

But neural networks need:
- x: A normalized tensor of pixel values
- y: An integer (0 to 199)

We need to transform our data!

### 3.1 The Transform Wrapper Class

In [ ]:
class TfmDS:
    """
    A wrapper that applies transforms to a dataset.
    
    This is a powerful pattern: instead of modifying our original dataset,
    we wrap it with a new class that applies transforms on-the-fly.
    
    Args:
        ds: The base dataset to wrap
        tfmx: Transform function for x (inputs/images)
        tfmy: Transform function for y (labels/targets)
    
    fc.noop is a function that does nothing (returns its input unchanged).
    We use it as the default so transforms are optional.
    """
    
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): 
        # Store the base dataset and transform functions
        self.ds = ds
        self.tfmx = tfmx
        self.tfmy = tfmy
    
    def __len__(self): 
        # Same length as the base dataset
        return len(self.ds)
    
    def __getitem__(self, i):
        # Get the raw data from the base dataset
        x, y = self.ds[i]
        # Apply transforms and return
        return self.tfmx(x), self.tfmy(y)

**Why use a wrapper instead of modifying the original class?**

1. **Flexibility**: We can easily swap out different transforms
2. **Reusability**: The same TfmDS works with any dataset
3. **Composability**: We can chain multiple wrappers together
4. **Clean code**: Each class has a single responsibility

### 3.2 Converting Class IDs to Integers

In [ ]:
# Load the list of class IDs from the wnids.txt file
# This file contains one WordNet ID per line, in a specific order
# Example content:
#   n01443537
#   n01629819
#   n01641577
#   ... (200 lines)
id2str = (path / 'wnids.txt').read_text().splitlines()

# Create a reverse mapping: string -> integer
# enumerate gives us (index, value) pairs: (0, 'n01443537'), (1, 'n01629819'), ...
# We flip them to create {string: index}
str2id = {v: k for k, v in enumerate(id2str)}

# Now str2id looks like:
# {
#     'n01443537': 0,
#     'n01629819': 1,
#     'n01641577': 2,
#     ... (200 entries)
# }

### 3.3 Image Normalization

Neural networks work best when inputs are **normalized** - meaning they have:
- Mean close to 0
- Standard deviation close to 1

**Why normalize?**

1. **Faster training**: Gradients are more balanced across features
2. **Better convergence**: Optimization landscape is smoother
3. **Consistent scale**: All inputs are in a similar range

**The formula:**

$$x_{normalized} = \frac{x - mean}{std}$$

If the original values have mean=0.5 and std=0.25, after normalization:
- New mean = (0.5 - 0.5) / 0.25 = 0
- New std = 0.25 / 0.25 = 1

In [ ]:
# Pre-computed mean and standard deviation for Tiny ImageNet
# These were calculated by averaging over all images in the training set
# One value per color channel (Red, Green, Blue)

xmean = tensor([0.47565, 0.40303, 0.31555])  # Mean for R, G, B
xstd = tensor([0.28858, 0.24402, 0.26615])   # Std for R, G, B

# Notice: Red channel has highest mean (0.476), Blue has lowest (0.316)
# This means Tiny ImageNet images tend to be slightly warm-toned (more red)

### 3.4 Defining the Transform Functions

In [ ]:
def tfmx(x):
    """
    Transform function for images.
    
    Takes a file path, returns a normalized tensor.
    
    Args:
        x: String path to an image file
    
    Returns:
        Tensor of shape (3, 64, 64) with normalized pixel values
    """
    # Step 1: Load the image from disk
    # read_image returns a tensor with shape (C, H, W) and values 0-255
    # C = channels (3 for RGB)
    # H = height (64 pixels)
    # W = width (64 pixels)
    # mode=ImageReadMode.RGB ensures we always get 3 channels
    img = read_image(x, mode=ImageReadMode.RGB)
    
    # Step 2: Convert from 0-255 integers to 0-1 floats
    img = img / 255
    
    # Step 3: Normalize by subtracting mean and dividing by std
    # The [:,None,None] trick:
    #   xmean has shape (3,)         - just 3 numbers
    #   xmean[:,None,None] has shape (3, 1, 1) - 3 numbers in a 3D structure
    #   This allows broadcasting with the (3, 64, 64) image
    #   Each channel gets its own mean/std applied
    normalized = (img - xmean[:, None, None]) / xstd[:, None, None]
    
    return normalized


def tfmy(y): 
    """
    Transform function for labels.
    
    Takes a string class ID, returns an integer tensor.
    
    Args:
        y: String class ID like 'n01443537'
    
    Returns:
        Tensor containing the integer class index (0-199)
    """
    # Look up the integer index in our mapping dictionary
    # Then wrap it in a tensor
    return tensor(str2id[y])

**Understanding Broadcasting with `[:,None,None]`:**

```python
# Our mean tensor:
xmean = tensor([0.47565, 0.40303, 0.31555])  # shape: (3,)

# Our image tensor (after dividing by 255):
img.shape  # (3, 64, 64)

# We can't directly subtract because shapes don't match:
# (3, 64, 64) - (3,) = ERROR!

# But if we add dimensions:
xmean[:, None, None].shape  # (3, 1, 1)

# Now PyTorch can "broadcast" - automatically repeat the values:
# (3, 64, 64) - (3, 1, 1) = (3, 64, 64)
# Each of the 3 mean values gets subtracted from all 64x64 pixels of its channel
```

In [ ]:
# Create transformed datasets by wrapping our raw datasets
tfm_tds = TfmDS(tds, tfmx, tfmy)  # Training data with transforms
tfm_vds = TfmDS(vds, tfmx, tfmy)  # Validation data with transforms

### 3.5 Denormalization for Visualization

To display images, we need to undo the normalization (convert back to 0-1 range).

In [ ]:
def denorm(x): 
    """
    Reverse the normalization for visualization.
    
    Reverses: x_original = x_normalized * std + mean
    Then clips to [0, 1] to ensure valid pixel values.
    
    Args:
        x: Normalized tensor
    
    Returns:
        Tensor with values in [0, 1] suitable for display
    """
    # Reverse the normalization formula
    result = x * xstd[:, None, None] + xmean[:, None, None]
    # Clip to valid range (some values might be slightly outside)
    return result.clip(0, 1)

### 3.6 Human-Readable Class Names

In [ ]:
# Load the mapping from WordNet IDs to human-readable names
# Format of words.txt: "n01443537\tgoldfish, Carassius auratus"
# Split by tab to get [ID, description]
all_synsets = [o.split('\t') for o in (path / 'words.txt').read_text().splitlines()]

# Create a dictionary of just our 200 classes
# For each (k, v) pair:
#   k = WordNet ID (e.g., 'n01443537')
#   v = description (e.g., 'goldfish, Carassius auratus')
# We take only the first name (before the comma) and only IDs in our dataset
synsets = {
    k: v.split(',', maxsplit=1)[0]  # Take first name only
    for k, v in all_synsets 
    if k in id2str  # Only include our 200 classes
}

# Now synsets looks like:
# {
#     'n01443537': 'goldfish',
#     'n01629819': 'European fire salamander',
#     ...
# }

### 3.7 Creating DataLoaders

In [ ]:
# Create DataLoaders - these load data in batches for training
# 
# get_dls is a helper function that creates train and validation DataLoaders
# Arguments:
#   tfm_tds: Training dataset
#   tfm_vds: Validation dataset
#   bs=512: Batch size (512 images per batch)
#   num_workers=8: Number of parallel processes for loading data
#
# DataLoaders is a container that holds both train and val loaders
# The * operator unpacks the tuple returned by get_dls

dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

**What are DataLoaders?**

A DataLoader takes a Dataset and:
1. **Batches** samples together (groups of 512)
2. **Shuffles** training data (so model sees different orders each epoch)
3. **Loads in parallel** using multiple CPU workers
4. **Transfers to GPU** when needed

```python
# Without DataLoader:
for i in range(len(dataset)):      # One sample at a time
    x, y = dataset[i]              # Load from disk (slow)
    train_on_single_sample(x, y)   # Inefficient

# With DataLoader:
for xb, yb in dataloader:          # Batch of 512 samples
    # xb shape: (512, 3, 64, 64)   # 512 images
    # yb shape: (512,)             # 512 labels
    train_on_batch(xb, yb)         # Efficient!
```

---
## Section 4: Data Augmentation

**Data augmentation** creates variations of training images to:
1. Artificially increase the size of our dataset
2. Teach the model to be robust to transformations
3. Reduce overfitting (memorizing training data)

Common augmentations for images:
- Random cropping (small translations)
- Horizontal flipping (left-right mirror)
- Color adjustments (brightness, contrast)
- Random erasing (cutout)

In [ ]:
def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop): 
    """
    Apply transforms to a batch of data.
    
    A batch is a tuple of (inputs, targets).
    This function applies separate transforms to each.
    
    Args:
        b: A batch tuple (inputs, targets)
        tfm_x: Transform for inputs (images)
        tfm_y: Transform for targets (labels)
    
    Returns:
        Transformed batch (transformed_inputs, transformed_targets)
    """
    return tfm_x(b[0]), tfm_y(b[1])

In [ ]:
# Define our augmentation pipeline as a sequence of transforms
# nn.Sequential chains them together: output of one is input to next

tfms = nn.Sequential(
    # T.Pad(4): Add 4 pixels of padding on each side
    # 64x64 image becomes 72x72
    # The padding is filled with zeros (black)
    T.Pad(4),
    
    # T.RandomCrop(64): Randomly crop back to 64x64
    # Since we padded by 4, the crop can be shifted up to 4 pixels in any direction
    # This teaches the model to handle small translations
    T.RandomCrop(64),
    
    # T.RandomHorizontalFlip(): 50% chance to flip left-right
    # Most objects look valid when flipped (cars, animals, etc.)
    # Exception: text, numbers - but we don't have those classes
    T.RandomHorizontalFlip(),
    
    # RandErase(): Randomly erase a rectangular region
    # Forces the model to not rely on any single region
    # Helps with occlusion (when objects are partially hidden)
    RandErase()
)

# Create a callback that applies augmentation to batches
# BatchTransformCB applies the transform during training
# partial(tfm_batch, tfm_x=tfms) creates a function that applies tfms to x only
# on_val=False means DON'T augment validation data (we want accurate evaluation)
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

**Visual Example of the Augmentation Pipeline:**

```
Original image (64x64):
+------------------+
|                  |
|    [cat face]    |
|                  |
+------------------+
        |
        v
After Pad(4) - now 72x72:
+----------------------+
|######################|
|##                  ##|
|##   [cat face]     ##|
|##                  ##|
|######################|
+----------------------+
  (# = black padding)
        |
        v
After RandomCrop(64) - back to 64x64:
+------------------+
|##                |
|##  [cat face]    |  <- Image shifted right & down
|##                |
+------------------+
        |
        v
After RandomHorizontalFlip (50% chance):
+------------------+
|                ##|
|    [ecaf tac]  ##|  <- Flipped!
|                ##|
+------------------+
        |
        v
After RandErase:
+------------------+
|                ##|
|    [e███ tac]  ##|  <- Random region erased
|                ##|
+------------------+
```

### 4.2 Activation Function and Weight Initialization

In [ ]:
# GeneralRelu: A modified version of the ReLU activation function
#
# Standard ReLU:  f(x) = max(0, x)
#   - Problem: "dead neurons" - if a neuron outputs negative, gradient is 0
#   - Problem: outputs are always positive (not centered around 0)
#
# Leaky ReLU:    f(x) = max(leak*x, x)  where leak is small (e.g., 0.1)
#   - For negative x, we get a small gradient instead of 0
#
# GeneralRelu:   f(x) = max(leak*x, x) - sub
#   - leak=0.1: Small slope for negative values
#   - sub=0.4: Subtract to center outputs around 0
#
# partial() creates a new function with some arguments pre-filled
# act_gr is now a function that creates GeneralRelu with leak=0.1, sub=0.4

act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

# init_weights: Initialize the weights of our neural network
# Good initialization is crucial for training deep networks
# leaky=0.1 tells it to use initialization appropriate for leaky ReLU
#
# iw is now a function that initializes weights for leaky ReLU

iw = partial(init_weights, leaky=0.1)

**Understanding `partial()`:**

```python
# Without partial, we'd have to write this every time:
activation = GeneralRelu(leak=0.1, sub=0.4)

# With partial, we create a "pre-configured" function:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

# Now act_gr() is equivalent to GeneralRelu(leak=0.1, sub=0.4)
activation = act_gr()

# This is useful when we need to pass a function (not an instance)
# to another function that will call it later
```

---
## Section 5: Building the ResNet Model

Now we'll build our neural network. We use a **ResNet** (Residual Network) architecture, which is one of the most successful architectures for image classification.

### 5.1 What is a ResNet?

A ResNet is made of **ResBlocks** (Residual Blocks). The key idea is the **skip connection**:

```
       input
         |
    +----+----+
    |         |
    v         |
 [Conv]       |
    |         |
    v         |
 [Conv]       |
    |         |
    +----+----+
         |
         v
        (+)   <- Add input to output
         |
       output
```

**Why skip connections help:**

1. **Gradient flow**: Gradients can flow directly through the skip connection
2. **Identity mapping**: If the convolutions learn "nothing", output = input
3. **Deeper networks**: Enables training of very deep networks (100+ layers)

### 5.2 First Model: Simple ResNet

In [ ]:
# Define the number of filters (channels) at each stage
# We start with 32 and double at each stage until 1024
# More channels = more "features" the model can learn
nfs = (32, 64, 128, 256, 512, 1024)

def get_dropmodel(act=act_gr, nfs=nfs, norm=nn.BatchNorm2d, drop=0.1):
    """
    Create a simple ResNet model with single blocks.
    
    Args:
        act: Activation function to use (default: GeneralRelu)
        nfs: Tuple of filter counts for each stage
        norm: Normalization layer type (default: BatchNorm2d)
        drop: Dropout probability (default: 0.1 = 10%)
    
    Returns:
        A complete neural network as nn.Sequential
    """
    layers = []
    
    # First layer: Initial convolution
    # 3 input channels (RGB) -> nfs[0] output channels (32)
    # kernel_size=5: 5x5 convolution (larger receptive field at start)
    # padding=2: keeps spatial size the same (64x64 -> 64x64)
    layers.append(nn.Conv2d(3, nfs[0], 5, padding=2))
    
    # ResBlocks with downsampling
    # Each ResBlock: nfs[i] channels -> nfs[i+1] channels
    # stride=2 means spatial size is halved (64->32->16->8->4->2)
    layers += [
        ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)
    ]
    
    # Global Average Pooling: (N, C, H, W) -> (N, C, 1, 1)
    # Takes the average of each channel across all spatial positions
    # This reduces our 2x2 feature maps to a single value per channel
    layers.append(nn.AdaptiveAvgPool2d(1))
    
    # Flatten: (N, 1024, 1, 1) -> (N, 1024)
    # Removes the spatial dimensions
    layers.append(nn.Flatten())
    
    # Dropout: Randomly set some values to 0 during training
    # drop=0.1 means 10% of values are zeroed
    # This helps prevent overfitting
    layers.append(nn.Dropout(drop))
    
    # Final classification layer
    # 1024 inputs (from last ResBlock) -> 200 outputs (one per class)
    # bias=False because we use BatchNorm which has its own bias
    layers.append(nn.Linear(nfs[-1], 200, bias=False))
    layers.append(nn.BatchNorm1d(200))
    
    # Combine all layers into a Sequential model
    # .apply(iw) initializes all weights using our init function
    return nn.Sequential(*layers).apply(iw)

**Network Architecture Visualization:**

```
Stage        Shape              Channels   What happens
-----        -----              --------   ------------
Input        (3, 64, 64)           3       RGB image
Conv 5x5     (32, 64, 64)         32       Initial feature extraction
ResBlock     (64, 32, 32)         64       Downsample (stride=2)
ResBlock     (128, 16, 16)       128       Downsample
ResBlock     (256, 8, 8)         256       Downsample
ResBlock     (512, 4, 4)         512       Downsample
ResBlock     (1024, 2, 2)       1024       Downsample
AvgPool      (1024, 1, 1)       1024       Global average
Flatten      (1024,)            1024       Remove spatial dims
Dropout      (1024,)            1024       Regularization
Linear+BN    (200,)              200       Class scores
```

### 5.3 Deeper Model with Multiple Blocks per Stage

In [ ]:
def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    """
    Create a sequence of ResNet blocks.
    
    This function creates multiple ResBlocks in sequence.
    Only the LAST block has the specified stride (for downsampling).
    Earlier blocks preserve spatial dimensions.
    
    Args:
        n_bk: Number of blocks to create
        ni: Number of input channels (for the first block)
        nf: Number of output channels (for all blocks)
        stride: Stride for the LAST block (default: 1, no downsampling)
        ks: Kernel size (default: 3x3)
        act: Activation function
        norm: Normalization layer type
    
    Returns:
        nn.Sequential containing n_bk ResBlocks
    
    Example with n_bk=3, ni=64, nf=128, stride=2:
        Block 0: 64 -> 128 channels, stride=1 (no downsampling)
        Block 1: 128 -> 128 channels, stride=1 (no downsampling)
        Block 2: 128 -> 128 channels, stride=2 (downsample!)
    """
    return nn.Sequential(*[
        ResBlock(
            # Input channels: first block uses ni, others use nf
            ni if i == 0 else nf,
            # Output channels: always nf
            nf,
            # Stride: only last block (i == n_bk-1) uses the specified stride
            stride=stride if i == n_bk - 1 else 1,
            ks=ks,
            act=act,
            norm=norm
        )
        for i in range(n_bk)  # Create n_bk blocks (i goes from 0 to n_bk-1)
    ])

In [ ]:
# Number of blocks at each stage
# More blocks at early stages (higher resolution) where we have more information
nbks = (3, 2, 2, 1, 1)  # 3 blocks, 2 blocks, 2 blocks, 1 block, 1 block

def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    """
    Create a deeper ResNet model with multiple blocks per stage.
    
    Args:
        act: Activation function
        nfs: Tuple of filter counts at each stage
        nbks: Tuple of block counts at each stage
        norm: Normalization layer type
        drop: Dropout probability
    
    Total blocks = 1 (initial) + sum(nbks) = 1 + 3+2+2+1+1 = 10 ResBlocks
    """
    layers = []
    
    # Initial ResBlock (instead of plain convolution)
    # 3 channels (RGB) -> nfs[0] channels (32)
    # ks=5 for larger initial receptive field
    # stride=1 keeps size (64x64)
    layers.append(ResBlock(3, nfs[0], ks=5, stride=1, act=act, norm=norm))
    
    # Multiple ResBlocks at each stage
    # For each stage i:
    #   - Create nbks[i] blocks
    #   - Input channels: nfs[i]
    #   - Output channels: nfs[i+1]
    #   - Last block in group has stride=2 (downsample)
    layers += [
        res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)  # 5 stages total
    ]
    
    # Classification head (same as before)
    layers += [
        nn.AdaptiveAvgPool2d(1),  # Global average pooling
        nn.Flatten(),              # Remove spatial dimensions
        nn.Dropout(drop)           # Regularization (20% dropout)
    ]
    
    # Final linear layer
    layers += [
        nn.Linear(nfs[-1], 200, bias=False),  # 1024 -> 200
        nn.BatchNorm1d(200)
    ]
    
    return nn.Sequential(*layers).apply(iw)

---
## Section 6: Training Setup

In [ ]:
# Define the optimizer
# AdamW is a variant of Adam with better weight decay (regularization)
# eps=1e-5: A small number added to prevent division by zero
#           (default is 1e-8, but 1e-5 works better with mixed precision)
opt_func = partial(optim.AdamW, eps=1e-5)

**What is an Optimizer?**

An optimizer updates the model's weights based on the gradients computed during backpropagation.

```
Basic gradient descent: weight = weight - learning_rate * gradient

Adam adds:
- Momentum: Use a running average of gradients (smoother updates)
- Adaptive learning rates: Different learning rates for different parameters

AdamW adds:
- Decoupled weight decay: Better regularization than standard L2
```

In [ ]:
# Define callbacks - functions that run at specific points during training
# Metrics callback: track accuracy during training
metrics = MetricsCB(accuracy=MulticlassAccuracy())

# Main callbacks list:
cbs = [
    DeviceCB(),              # Move data to GPU automatically
    metrics,                 # Track and display metrics
    ProgressCB(plot=True),   # Show progress bar and loss plot
    MixedPrecision()         # Use FP16 for faster training
]

# Training hyperparameters
epochs = 25        # Number of times to go through the entire dataset
lr = 3e-2          # Learning rate (0.03) - how big our weight updates are

# Calculate total training steps (for learning rate scheduler)
# Each epoch has len(dls.train) batches
tmax = epochs * len(dls.train)

# Learning rate scheduler: OneCycleLR
# This gradually changes the learning rate during training:
# 1. Warmup: LR increases from small to max_lr
# 2. Annealing: LR decreases from max_lr to small
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Extra callbacks: scheduler and augmentation
xtra = [BatchSchedCB(sched), augcb]

# Create the Learner (combines model, data, loss function, and training loop)
learn = Learner(
    get_dropmodel(),       # The model
    dls,                   # The data
    F.cross_entropy,       # Loss function for classification
    lr=lr,                 # Learning rate
    cbs=cbs+xtra,          # All our callbacks
    opt_func=opt_func      # The optimizer
)

**What is Mixed Precision Training?**

Normally, neural networks use 32-bit floating point numbers (FP32). Mixed precision uses 16-bit (FP16) for most operations:

| Aspect | FP32 | FP16 |
|--------|------|------|
| Memory per number | 4 bytes | 2 bytes |
| Training speed | Baseline | ~2x faster |
| GPU memory | Baseline | ~50% less |
| Precision | High | Good enough |

The `MixedPrecision()` callback automatically handles the conversion, keeping master weights in FP32 for stability.

---
## Section 7: Advanced Data Augmentation

Now we add **TrivialAugmentWide** - a simple but very effective augmentation strategy.

In [ ]:
# Advanced augmentation pipeline including TrivialAugmentWide
aug_tfms = nn.Sequential(
    T.Pad(4),                   # Pad for random crop
    T.RandomCrop(64),           # Random crop
    T.RandomHorizontalFlip(),   # Random flip
    T.TrivialAugmentWide()      # Random augmentation from a predefined set
)

# Normalization transform (using torchvision's Normalize)
norm_tfm = T.Normalize(xmean, xstd)

# Random erase transform
erase_tfm = RandErase()

**What is TrivialAugmentWide?**

Instead of carefully designing an augmentation policy, TrivialAugmentWide randomly picks ONE augmentation from a set and applies it with a random strength:

**Possible augmentations:**
- Identity (no change)
- AutoContrast
- Equalize
- Rotate
- Solarize
- Color
- Posterize
- Contrast
- Brightness
- Sharpness
- ShearX, ShearY
- TranslateX, TranslateY

**Why it works:**
- Simple to implement (no hyperparameter search)
- Diverse augmentations prevent overfitting
- Only ONE augmentation per image = not too destructive

In [ ]:
# Import PIL for image handling
# TrivialAugmentWide works with PIL Images, not tensors
from PIL import Image

def tfmx(x, aug=False):
    """
    Transform function for images with optional augmentation.
    
    This function handles the entire pipeline:
    1. Load image from disk
    2. Apply augmentation (if training)
    3. Convert to tensor
    4. Normalize
    5. Apply random erase (if training)
    
    Args:
        x: Path to image file
        aug: Whether to apply augmentation (True for training, False for validation)
    
    Returns:
        Normalized tensor ready for the model
    """
    # Step 1: Load image as PIL Image
    # TrivialAugmentWide requires PIL Image format
    # .convert('RGB') ensures we have 3 channels even for grayscale images
    x = Image.open(x).convert('RGB')
    
    # Step 2: Apply augmentation (training only)
    if aug: 
        x = aug_tfms(x)
    
    # Step 3: Convert PIL Image to tensor
    # TF.to_tensor converts (H, W, C) PIL Image to (C, H, W) tensor in [0, 1]
    x = TF.to_tensor(x)
    
    # Step 4: Normalize using mean and std
    x = norm_tfm(x)
    
    # Step 5: Apply random erase (training only)
    # RandErase expects a batch, so we add and remove a batch dimension
    # x[None] adds batch dim: (C,H,W) -> (1,C,H,W)
    # [0] removes it: (1,C,H,W) -> (C,H,W)
    if aug: 
        x = erase_tfm(x[None])[0]
    
    return x

In [ ]:
# Create new transformed datasets
# Training: aug=True (apply augmentation)
# Validation: aug=False (no augmentation - we want consistent evaluation)
tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)  # aug defaults to False

# Recreate DataLoaders with new transforms
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

---
## Section 8: Pre-activation ResNet

Now we implement a **Pre-activation ResNet** - a variant where batch normalization and activation come BEFORE the convolution instead of after.

### 8.1 Standard vs Pre-activation Order

**Standard ResNet (Post-activation):**
```
Conv -> BatchNorm -> ReLU
```

**Pre-activation ResNet:**
```
BatchNorm -> ReLU -> Conv
```

**Why pre-activation is often better:**

1. **Better gradient flow**: The skip connection adds "clean" features directly
2. **Easier optimization**: Acts like training multiple shallow networks
3. **Better regularization**: BatchNorm before conv regularizes better

In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    """
    Create a pre-activation convolution block.
    
    Order: [Norm] -> [Activation] -> Conv
    
    Args:
        ni: Number of input channels
        nf: Number of output channels
        ks: Kernel size (default: 3x3)
        stride: Stride (default: 1, no downsampling)
        act: Activation function class (default: ReLU)
        norm: Normalization layer class (default: None)
        bias: Whether to use bias in conv (default: True)
    
    Returns:
        nn.Sequential containing the layers
    """
    layers = []
    
    # Step 1: Normalization (if specified)
    # Note: norm(ni) normalizes INPUT channels, not output
    if norm: 
        layers.append(norm(ni))
    
    # Step 2: Activation (if specified)
    # act() creates an instance of the activation class
    if act: 
        layers.append(act())
    
    # Step 3: Convolution
    # padding=ks//2 keeps spatial size the same (for stride=1)
    # e.g., ks=3 -> padding=1, ks=5 -> padding=2
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    
    return nn.Sequential(*layers)


def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    """
    Create a double convolution block (two pre-activation convs).
    
    This is the "body" of a ResBlock:
    - First conv: ni -> nf, stride=1
    - Second conv: nf -> nf, with specified stride
    
    Args:
        ni: Input channels
        nf: Output channels
        stride: Stride for the second convolution
        act: Activation function
        norm: Normalization layer
        ks: Kernel size
    """
    return nn.Sequential(
        # First conv: change channels, keep spatial size
        conv(ni, nf, stride=1, act=act, norm=norm, ks=ks),
        # Second conv: keep channels, maybe downsample
        conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks)
    )

In [ ]:
class ResBlock(nn.Module):
    """
    Pre-activation Residual Block.
    
    Architecture:
    
                input
                  |
        +---------+---------+
        |                   |
        v                   v
    [_conv_block]      [idconv + pool]
    (main path)        (skip connection)
        |                   |
        +---------+---------+
                  |
                  v
                 (+) addition
                  |
                output
    
    Key features:
    - Pre-activation: BN and ReLU come before conv
    - Average pooling for downsampling (not strided conv)
    - 1x1 conv if channels change
    """
    
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        """
        Initialize the ResBlock.
        
        Args:
            ni: Number of input channels
            nf: Number of output channels
            stride: Stride for downsampling (1=no downsample, 2=halve size)
            ks: Kernel size for convolutions
            act: Activation function
            norm: Normalization layer type
        """
        super().__init__()  # Initialize parent class (nn.Module)
        
        # Main path: two convolutions
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        
        # Skip connection path:
        # If channels change (ni != nf), we need a 1x1 conv to match dimensions
        # fc.noop is a function that returns its input unchanged
        self.idconv = fc.noop if ni == nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        
        # Downsampling for skip connection:
        # If stride > 1, main path downsamples, so skip connection must too
        # We use AvgPool instead of strided conv (preserves more information)
        # ceil_mode=True rounds up when size is odd (ensures dimensions match)
        self.pool = fc.noop if stride == 1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): 
        """
        Forward pass through the block.
        
        Args:
            x: Input tensor of shape (batch, ni, height, width)
        
        Returns:
            Output tensor of shape (batch, nf, height//stride, width//stride)
        """
        # Main path: apply convolutions
        main = self.convs(x)
        
        # Skip path: optionally pool then optionally 1x1 conv
        skip = self.idconv(self.pool(x))
        
        # Add them together (the key to ResNets!)
        return main + skip

**Why use Average Pooling instead of Strided Convolution?**

When downsampling the skip connection:

| Method | How it works | Information loss |
|--------|--------------|------------------|
| Strided 1x1 conv | Takes every other pixel | 75% of pixels ignored |
| Average pooling | Averages 2x2 blocks | All pixels contribute |

Average pooling is generally better because no information is discarded.

### 8.2 Updated Model Function

In [ ]:
def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    """
    Create a pre-activation ResNet model.
    
    Since we use pre-activation (BN-ReLU-Conv), we need to add
    explicit activation and normalization at the end of the network
    (the last conv doesn't have activation after it by default).
    
    Args:
        act: Activation function
        nfs: Tuple of filter counts per stage
        nbks: Tuple of block counts per stage
        norm: Normalization layer type
        drop: Dropout probability
    
    Returns:
        Complete model as nn.Sequential
    """
    layers = []
    
    # Initial conv (no pre-activation - there's nothing to normalize yet)
    # 3 RGB channels -> nfs[0] (32) channels
    # 5x5 kernel for larger initial receptive field
    layers.append(nn.Conv2d(3, nfs[0], 5, padding=2))
    
    # Pre-activation ResBlocks at each stage
    layers += [
        res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)
    ]
    
    # IMPORTANT: Final activation and normalization
    # Because pre-activation puts BN-ReLU BEFORE conv,
    # the last conv's output hasn't been activated yet
    layers.append(act_gr())              # Final activation
    layers.append(norm(nfs[-1]))         # Final normalization
    
    # Classification head
    layers.append(nn.AdaptiveAvgPool2d(1))  # Global average pooling
    layers.append(nn.Flatten())             # Remove spatial dimensions
    layers.append(nn.Dropout(drop))         # Regularization
    
    # Final classification layer
    layers.append(nn.Linear(nfs[-1], 200, bias=False))
    layers.append(nn.BatchNorm1d(200))
    
    return nn.Sequential(*layers).apply(iw)

---
## Section 9: Training the Widish Model

Now we create and train our "widish" model - wider than standard but not as wide as the full "wide" version.

In [ ]:
# Training configuration
epochs = 50    # Train for 50 epochs (more than before for better accuracy)
lr = 0.1       # Higher learning rate (works well with pre-activation ResNets)

# Total training steps for the scheduler
tmax = epochs * len(dls.train)

# OneCycleLR scheduler
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Extra callbacks (no augcb needed - augmentation is in tfmx now)
xtra = [BatchSchedCB(sched)]

# ============================================================
# THE "WIDISH" MODEL CONFIGURATION
# ============================================================
#
# Compare different configurations:
#
# Standard:  nfs=(32, 64, 128, 256, 512, 1024)
#            nbks=(3, 2, 2, 1, 1)
#
# Widish:    nfs=(32, 64, 128, 512, 768, 1024)
#            nbks=(1, 2, 4, 2, 2)
#                         ↑ ↑  ↑
#            More channels in middle stages
#            (128->512 jumps 4x instead of 2x)
#
# Wide:      nfs=(32, 64, 128, 512, 1024, 1536)
#            nbks=(1, 2, 8, 2, 2)
#            Even more channels, even more blocks

model = get_dropmodel(
    nbks=(1, 2, 4, 2, 2),                   # Block distribution: 1+2+4+2+2 = 11 blocks
    nfs=(32, 64, 128, 512, 768, 1024),      # Channel widths (widish configuration)
    drop=0.1                                # 10% dropout
)

# Create the learner
learn = Learner(
    model,               # Our widish model
    dls,                 # DataLoaders
    F.cross_entropy,     # Loss function
    lr=lr,               # Learning rate
    cbs=cbs+xtra,        # Callbacks
    opt_func=opt_func    # Optimizer
)

**Understanding the "Widish" Configuration:**

```
Stage    Standard    Widish      Difference
-----    --------    ------      ----------
0           32          32       Same
1           64          64       Same
2          128         128       Same
3          256         512       2x wider!
4          512         768       1.5x wider
5         1024        1024       Same

Block distribution:
Stage    Standard    Widish
-----    --------    ------
0-1         3           1        Fewer early blocks
1-2         2           2        Same
2-3         2           4        More blocks here!
3-4         1           2        More blocks
4-5         1           2        More blocks
Total       9          11        Widish has more blocks
```

**Why "widish" instead of fully "wide"?**

- Uses less GPU memory than fully wide
- Faster to train
- Still gets good accuracy
- Good compromise between standard and wide

In [ ]:
# Train the model for 50 epochs
# This will take a while - expect ~20-30 minutes on a good GPU
learn.fit(epochs)

**What to expect during training:**

- Training loss should decrease steadily
- Validation loss should follow but may plateau or increase slightly (overfitting)
- Accuracy should reach ~55-60% (Tiny ImageNet is challenging!)
- Learning rate will follow the OneCycle pattern (increase then decrease)

In [ ]:
# Save the trained model
# This saves all weights so we can load them later without retraining
torch.save(learn.model, 'models/inettiny-widish-50')

---
## Summary

### What We Built

A **pre-activation ResNet** classifier with a "widish" configuration:

| Component | Description |
|-----------|-------------|
| Dataset | Tiny ImageNet (200 classes, 64x64 images) |
| Architecture | Pre-activation ResNet (BN-ReLU-Conv) |
| Channel widths | (32, 64, 128, 512, 768, 1024) |
| Block counts | (1, 2, 4, 2, 2) = 11 blocks total |
| Augmentation | TrivialAugmentWide + RandomCrop + RandomFlip + RandErase |
| Optimizer | AdamW with eps=1e-5 |
| Scheduler | OneCycleLR (max_lr=0.1) |
| Training | 50 epochs with mixed precision |

### Key Concepts Learned

1. **Custom Datasets**: How to create PyTorch datasets for any data format
2. **Transform Wrappers**: Flexible way to apply transforms to datasets
3. **Image Normalization**: Why and how to normalize images
4. **Data Augmentation**: Techniques to prevent overfitting
5. **ResNet Architecture**: Skip connections for training deep networks
6. **Pre-activation**: BN-ReLU-Conv order for better gradient flow
7. **Wide Networks**: More channels vs more layers trade-off
8. **Modern Training**: OneCycleLR, mixed precision, AdamW

### fin -